# CineMatrix — Movie Recommendations via Matrix Factorization (From Scratch)

**Hack-o-Week Weeks 5–6 project**
Topics applied: vectors, matrices, dot products, eigenvalues (intuition), derivatives, gradients, chain rule.

We will build a recommender system that predicts how a user would rate a movie they haven't seen,
using **matrix factorization** trained with **gradient descent that we derive and code ourselves**
(no `sklearn.decomposition`, no `surprise`, no autograd — just NumPy).

**Dataset:** MovieLens 100K — https://www.kaggle.com/datasets/prajitdatta/movielens-100k-dataset
Download it and place `u.data` and `u.item` in the `../data/` folder (see `data/README.md`).


## 1. The problem, in matrix form

We have a set of users and a set of movies. Some (user, movie) pairs have a rating (1–5 stars);
most don't (nobody has watched/rated every movie). Arrange this as a matrix **R**:

$$
R \in \mathbb{R}^{n_{users} \times n_{movies}}
$$

where `R[u, i]` is user *u*'s rating of movie *i*, and `0` means "unobserved" (not "hated it").

Our goal: fill in the blanks. To do that we assume every user and every movie can be described by
a short **vector** of *k* hidden ("latent") taste dimensions — think genre-leaning, pacing preference,
etc., except the model discovers these dimensions itself; we never name them.

$$
R \approx P \cdot Q^{T}, \qquad P \in \mathbb{R}^{n_{users}\times k},\;\; Q \in \mathbb{R}^{n_{movies}\times k}
$$

Predicting a rating for user *u*, movie *i* is just a **dot product** of two vectors:

$$
\hat{r}(u,i) = p_u \cdot q_i = \sum_{f=1}^{k} P_{u,f}\,Q_{i,f}
$$


In [1]:
import sys
sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt

from data_loader import load_ratings_matrix, load_movie_titles, train_test_split_matrix
from matrix_factorization import MatrixFactorization
from visualize import plot_loss_curve, plot_movie_landscape, eigen_project_2d
from recommend import recommend_for_user

np.random.seed(42)


## 2. Load the data

`R` is our matrix of vectors: each row is a user's rating vector across all movies (mostly zeros —
this is a *sparse* matrix problem, which is exactly why we can't just invert it or take a plain
eigendecomposition of R itself).

In [2]:
RATINGS_PATH = "../data/u.data"
ITEMS_PATH = "../data/u.item"

R, user_map, movie_map = load_ratings_matrix(RATINGS_PATH)
titles_map = load_movie_titles(ITEMS_PATH)

n_users, n_movies = R.shape
n_observed = np.count_nonzero(R)
sparsity = 100 * (1 - n_observed / (n_users * n_movies))

print(f"Users: {n_users}, Movies: {n_movies}")
print(f"Observed ratings: {n_observed}")
print(f"Matrix sparsity: {sparsity:.2f}% empty")


FileNotFoundError: [Errno 2] No such file or directory: '../data/u.data'

## 3. Train / test split

We hide 20% of the *observed* ratings so we can check later whether the model generalizes, rather
than just memorizing the training ratings.

In [ ]:
R_train, test_triples = train_test_split_matrix(R, test_ratio=0.2, seed=42)
print(f"Training ratings: {np.count_nonzero(R_train)}")
print(f"Held-out test ratings: {len(test_triples)}")


## 4. Deriving the gradients (the calculus part)

For one observed rating `r(u,i)`, define the error:

$$
e(u,i) = r(u,i) - \hat r(u,i) = r(u,i) - p_u \cdot q_i
$$

and the (regularized) loss contributed by that single rating:

$$
L_{u,i} = e(u,i)^2 \;+\; \lambda\left(\lVert p_u \rVert^2 + \lVert q_i \rVert^2\right)
$$

`e(u,i)` is itself a function of `p_u` (and of `q_i`), so to get `∂L/∂p_u` we need the **chain rule**:

$$
\frac{\partial L_{u,i}}{\partial p_u}
= \frac{\partial L_{u,i}}{\partial e}\cdot\frac{\partial e}{\partial p_u}
= 2e(u,i)\cdot(-q_i) \;+\; 2\lambda p_u
$$

$$
\frac{\partial L_{u,i}}{\partial q_i}
= 2e(u,i)\cdot(-p_u) \;+\; 2\lambda q_i
$$

Folding the constant 2 into the learning rate `η` gives the classic **stochastic gradient descent**
update (this is exactly what `MatrixFactorization.fit()` in `src/matrix_factorization.py` does,
one observed rating at a time):

$$
p_u \leftarrow p_u + \eta\,\big(e(u,i)\,q_i - \lambda p_u\big)
$$
$$
q_i \leftarrow q_i + \eta\,\big(e(u,i)\,p_u - \lambda q_i\big)
$$

Each update nudges both vectors a small step **downhill** on the loss surface — the same core idea
behind backpropagation in neural networks, just applied to a much simpler (2-layer, bilinear) model.

## 5. Train the model

In [ ]:
mf = MatrixFactorization(
    n_factors=20,
    learning_rate=0.01,
    reg=0.02,
    n_epochs=40,
    seed=42,
    verbose=True,
)

mf.fit(R_train)


## 6. Convergence — is gradient descent actually working?

In [ ]:
plot_loss_curve(mf.loss_history, save_path="../outputs/loss_curve.png")


## 7. Evaluate on held-out ratings

RMSE (root-mean-squared-error) tells us, on average, how far off our predicted star rating is from
the true rating the user actually gave (on unseen data).

In [ ]:
test_rmse = mf.rmse(test_triples)
print(f"Test RMSE: {test_rmse:.4f}  (out of a 1-5 rating scale)")


## 8. Eigenvalues in action — visualizing the learned taste-space

We take the movie latent matrix `Q` (n_movies × k) and project it down to 2D using the top-2
**eigenvectors** of its covariance matrix `QᵀQ` — the same idea behind PCA. The corresponding
**eigenvalues** tell us how much of the "taste variance" each axis explains.

If the factorization learned something meaningful, similar movies (same genre / vibe) should
cluster together in this eigenvector-projected space.

In [ ]:
movie_ids = [None] * n_movies
for mid, idx in movie_map.items():
    movie_ids[idx] = mid

plot_movie_landscape(mf.Q, movie_ids, titles_map, n_labels=20,
                      save_path="../outputs/movie_landscape.png")


## 9. Connecting to true SVD (bonus / stretch goal)

Our gradient-descent factorization is an *approximation* to the Singular Value Decomposition (SVD)
of R, adapted to handle missing entries (true SVD needs a dense matrix). Let's compare against
`numpy.linalg.svd` on a **mean-filled** version of R, and look at the eigenvalues of `RᵀR`
(which equal the squared singular values of R) to see how much variance the top factors capture.

In [ ]:
# Fill missing entries with each movie's mean rating (a common, simple baseline)
R_filled = R_train.copy()
movie_means = np.true_divide(R_filled.sum(axis=0), (R_filled != 0).sum(axis=0) + 1e-9)
zero_mask = R_filled == 0
R_filled[zero_mask] = np.take(movie_means, np.nonzero(zero_mask)[1])

# True SVD
U, S, Vt = np.linalg.svd(R_filled, full_matrices=False)

# Eigenvalues of R^T R equal S^2 (squared singular values)
eigenvalues_RtR = S ** 2
explained_variance = eigenvalues_RtR / eigenvalues_RtR.sum()

plt.figure(figsize=(7, 4))
plt.plot(np.cumsum(explained_variance[:50]), marker="o", markersize=3)
plt.xlabel("Number of singular values / eigenvalues kept")
plt.ylabel("Cumulative variance explained")
plt.title("How many latent dimensions do we actually need?")
plt.grid(alpha=0.3)
plt.show()

print(f"Top 20 components explain {explained_variance[:20].sum()*100:.1f}% of variance")


## 10. Get recommendations for a real user

Recommending is just: take a user's latent vector `p_u`, dot it with **every** movie's latent
vector (`P[u] @ Q.T`), and rank movies by predicted score — excluding movies they've already rated.

In [ ]:
EXAMPLE_USER_ID = 1  # original user id from the dataset
user_row = user_map[EXAMPLE_USER_ID]

recs = recommend_for_user(mf, user_row, R_train, movie_map, titles_map, top_n=10)

print(f"Top 10 recommendations for user {EXAMPLE_USER_ID}:\n")
for rank, (title, score) in enumerate(recs, start=1):
    print(f"{rank:2d}. {title}  (predicted rating: {score:.2f})")


## 11. Wrap-up

- **Vectors**: each user and movie is a learned point in k-dimensional taste space.
- **Matrices**: R, P, Q; predictions are a full matrix multiplication `P @ Qᵀ`.
- **Dot product**: the literal prediction formula for a single rating.
- **Eigenvalues**: used to find the "most important" directions in the learned taste-space (PCA
  on `Q`) and to connect our factorization to the theoretical ceiling given by true SVD.
- **Derivatives / Gradients / Chain rule**: derived by hand in Section 4, and used to actually
  train the whole model in Section 5.

### Ideas to extend further
- Add bias terms `b_u`, `b_i` and a global mean `μ` — derive the extra gradient terms yourself.
- Try different `k` (number of latent factors) and plot RMSE vs. k.
- Build a tiny web UI (Streamlit/Gradio) around `recommend.py` for your demo.
